# 02 — Conditional Independence Testing

**Companion to `docs/02_conditional_independence.md` (Phase 1).**

CMI tells us *how much* dependence there is; a **CI test** decides whether that
dependence is statistically significant given finite data. We build the
**G-test** from scratch: `G = 2N·Î(X;Y|Z)`, which is χ²-distributed under the
null `X ⊥ Y | Z`.

| p-value | conclusion |
|---------|-----------|
| `> α`   | fail to reject → treat as **independent** |
| `≤ α`   | reject → treat as **dependent** |


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
rng = np.random.default_rng(0)
ALPHA = 0.05

## 0. The G-test, from scratch

For each configuration (stratum) of the conditioning set `Z`, build the X×Y
contingency table, compare observed to expected-under-independence counts, and
sum the likelihood-ratio statistic. Degrees of freedom use only the rows/cols
that actually occur in each stratum (**adjusted dof**).


In [3]:
def _table(x, y):
    xs, xi = np.unique(x, return_inverse=True)
    ys, yi = np.unique(y, return_inverse=True)
    tab = np.zeros((len(xs), len(ys)))
    np.add.at(tab, (xi, yi), 1)
    return tab

def _strata_tables(x, y, z):
    if not z:
        yield _table(x, y); return
    zmat = np.column_stack([np.asarray(c).reshape(-1) for c in z])
    _, inv = np.unique(zmat, axis=0, return_inverse=True)
    for s in range(inv.max() + 1):
        m = inv == s
        yield _table(x[m], y[m])

def g_test(x, y, z=None):
    """Return (G statistic, dof, p_value) for H0: X ⊥ Y | Z."""
    z = z or []
    G, dof = 0.0, 0
    for tab in _strata_tables(np.asarray(x), np.asarray(y), z):
        n = tab.sum()
        if n == 0:
            continue
        row = tab.sum(1, keepdims=True); col = tab.sum(0, keepdims=True)
        expected = row @ col / n
        nz = (tab > 0) & (expected > 0)
        G += 2.0 * np.sum(tab[nz] * np.log(tab[nz] / expected[nz]))
        r = int(np.count_nonzero(row)); cc = int(np.count_nonzero(col))
        dof += max(r-1, 0) * max(cc-1, 0)
    dof = max(dof, 1)
    return float(G), dof, float(chi2.sf(G, dof))

## 1. A small network to test on

`A → B → C`, plus independent noise `D`.


In [4]:
n = 8000
A = rng.integers(0, 2, n)
B = (A ^ (rng.random(n) < 0.15)).astype(int)
C = (B ^ (rng.random(n) < 0.15)).astype(int)
D = rng.integers(0, 2, n)
df = pd.DataFrame({"A": A, "B": B, "C": C, "D": D})

def report(x, y, z=None):
    z = z or []
    G, dof, p = g_test(df[x].to_numpy(), df[y].to_numpy(), [df[v].to_numpy() for v in z])
    z_str = f" | {z}" if z else ""
    verdict = "INDEPENDENT" if p > ALPHA else "dependent"
    print(f"test({x};{y}{z_str}):  G={G:8.2f}  dof={dof:<3} p={p:.3e}  -> {verdict}")

df.head()

,A,B,C,D
0,1,1,1,1
1,1,1,1,1
2,1,1,1,0
3,0,0,0,0
4,0,0,0,1


## 2. Marginal tests

In [5]:
report("A", "B")
report("B", "C")
report("A", "C")
report("A", "D")

test(A;B):  G= 4321.25  dof=1   p=0.000e+00  -> dependent
test(B;C):  G= 4477.13  dof=1   p=0.000e+00  -> dependent
test(A;C):  G= 2065.07  dof=1   p=0.000e+00  -> dependent
test(A;D):  G=    0.14  dof=1   p=7.037e-01  -> INDEPENDENT


## 3. The key flip: condition on B

`A` and `C` are dependent — but conditioning on the mediator `B` should make
them independent. The whole MB machinery depends on a CI test getting this right.


In [6]:
report("A", "C")            # dependent
report("A", "C", ["B"])     # should flip to INDEPENDENT

test(A;C):  G= 2065.07  dof=1   p=0.000e+00  -> dependent
test(A;C | ['B']):  G=    0.01  dof=2   p=9.955e-01  -> INDEPENDENT


## 4. Power vs sample size

A CI test only detects dependence with enough data. Watch a *weak* dependence's
p-value fall below α as N grows; too little data → false negative.


In [7]:
for n in [50, 200, 1000, 5000, 20000]:
    a = rng.integers(0, 2, n)
    b = np.where(rng.random(n) < 0.60, a, 1 - a)    # weak dependence
    _, _, p = g_test(a, b)
    print(f"N={n:>6}:  p={p:.3e}  -> {'INDEPENDENT' if p > ALPHA else 'dependent'}")

N=    50:  p=1.238e-01  -> INDEPENDENT
N=   200:  p=3.087e-02  -> dependent
N=  1000:  p=4.935e-11  -> dependent
N=  5000:  p=6.789e-46  -> dependent
N= 20000:  p=2.391e-182  -> dependent


## 5. The cost of a large conditioning set

Each variable added to `Z` multiplies the strata, emptying cells and
destabilizing the test even when nothing is going on. Condition two independent
variables on a growing set of irrelevant randoms and watch dof explode.


In [8]:
n = 4000
A0 = rng.integers(0, 2, n); D0 = rng.integers(0, 2, n)
for k in range(0, 6):
    Z = [rng.integers(0, 2, n) for _ in range(k)]
    G, dof, p = g_test(A0, D0, Z)
    print(f"|Z|={k}:  dof={dof:>4}  G={G:8.2f}  p={p:.3f}")

|Z|=0:  dof=   1  G=    1.55  p=0.213
|Z|=1:  dof=   2  G=    1.77  p=0.414
|Z|=2:  dof=   4  G=    4.35  p=0.360
|Z|=3:  dof=   8  G=    9.45  p=0.306
|Z|=4:  dof=  16  G=   13.66  p=0.624
|Z|=5:  dof=  32  G=   31.75  p=0.479


### Takeaways

- A CI test turns a CMI estimate into a yes/no decision via a p-value and α.
- Conditioning on a mediator flips dependence → independence.
- Detecting weak dependence needs enough samples; small N → false negatives.
- Large `|Z|` inflates dof and thins the data → unreliable tests. **This is why
  IAMB / HITON keep conditioning sets small.**

**Exercises:** `exercises/02_conditional_independence.md`.
